<a href="https://colab.research.google.com/github/AriesConsulting/808s-Auto-Generations-/blob/main/Cinematic_VFX_Pipeline_Orchestrator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import json
import time
import hashlib
import logging
import datetime
from pathlib import Path
from typing import Callable, Dict, Any, List

# --- Production Library Imports (with graceful stubs for instant execution) ---
try:
    import numpy as np
except ImportError:
    print("Error: numpy is required. Please 'pip install numpy'")
    exit(1)

try:
    import imageio.v3 as imageio
    HAS_IMAGEIO = True
except ImportError:
    HAS_IMAGEIO = False

try:
    import onnxruntime as ort
    HAS_ORT = True
except ImportError:
    HAS_ORT = False

try:
    import opentimelineio as otio
    HAS_OTIO = True
except ImportError:
    HAS_OTIO = False

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')

# ==========================================
# 1. CORE PIPELINE ARCHITECTURE
# ==========================================

class Pipeline:
    def __init__(self, name: str, stages: List[Callable[[Dict[str, Any]], Dict[str, Any]]]):
        self.name = name
        self.stages = stages

    def run(self, context: Dict[str, Any]) -> Dict[str, Any]:
        logging.info(f"--- Starting Pipeline: {self.name} for {context.get('shot_name', 'Unknown')} ---")
        for stage in self.stages:
            stage_name = stage.__name__
            start_time = time.time()

            try:
                context = stage(context)
                elapsed = time.time() - start_time
                logging.info(f"  [+] Stage Complete: {stage_name} ({elapsed:.3f}s)")
            except Exception as e:
                logging.error(f"  [!] Stage FAILED: {stage_name} -> {str(e)}")
                context['status'] = 'FAILED'
                context['error'] = str(e)
                break

        if context.get('status') != 'FAILED':
            context['status'] = 'SUCCESS'
        return context

# ==========================================
# 2. PIPELINE STAGES
# ==========================================

def ingest_exr_stage(ctx: Dict[str, Any]) -> Dict[str, Any]:
    """Reads camera original, converts to linear float32, hashes original file."""
    src = ctx['input_path']
    if not os.path.exists(src):
        raise FileNotFoundError(f"Missing source file: {src}")

    # Generate cryptographic hash of input for provenance
    with open(src, 'rb') as f:
        ctx['input_hash'] = hashlib.sha256(f.read()).hexdigest()

    # Load Image
    if HAS_IMAGEIO:
        img = imageio.imread(src)
        # Normalize to float32 [0.0, 1.0] representing linear light
        if img.dtype == np.uint8:
            ctx['frame'] = img.astype(np.float32) / 255.0
        elif img.dtype == np.uint16:
            ctx['frame'] = img.astype(np.float32) / 65535.0
        else:
            ctx['frame'] = img.astype(np.float32)
    else:
        # Fallback dummy data if no imageio
        ctx['frame'] = np.random.rand(1080, 1920, 3).astype(np.float32)

    # Initialize AOVs (Arbitrary Output Variables)
    ctx['aovs'] = {}
    return ctx

def ml_roto_stage(ctx: Dict[str, Any]) -> Dict[str, Any]:
    """Runs segmentation (e.g., Segment Anything) to generate a subject matte."""
    frame = ctx['frame']
    model_path = ctx.get('roto_model_path', 'models/sam_vit_h.onnx')

    # Mock ONNX Execution
    if HAS_ORT and os.path.exists(model_path):
        session = ort.InferenceSession(model_path)
        # Real execution would go here
        matte = np.zeros(frame.shape[:2], dtype=np.float32)
    else:
        # Mock behavior: Create a synthetic matte (a circle in the middle)
        h, w = frame.shape[:2]
        y, x = np.ogrid[:h, :w]
        mask = ((x - w/2)**2 + (y - h/2)**2 <= (h/4)**2)
        matte = mask.astype(np.float32)

    ctx['aovs']['roto_matte'] = matte
    ctx['metadata']['roto_model'] = "stub_sam_v1"

    # Flag for manual sidecar review
    ctx['requires_human_review'] = True
    return ctx

def ml_denoise_stage(ctx: Dict[str, Any]) -> Dict[str, Any]:
    """Applies Neural Denoising to the main frame."""
    frame = ctx['frame']
    model_path = ctx.get('denoise_model_path', 'models/real_esrgan.onnx')

    if HAS_ORT and os.path.exists(model_path):
        session = ort.InferenceSession(model_path)
        input_name = session.get_inputs()[0].name

        # Format NCHW for PyTorch/ONNX
        inp = np.transpose(frame, (2, 0, 1))
        inp = np.expand_dims(inp, axis=0).astype(np.float32)

        out = session.run(None, {input_name: inp})[0]
        # Format back to HWC
        denoised = np.transpose(out[0], (1, 2, 0))
        ctx['frame'] = np.clip(denoised, 0.0, 1.0)
    else:
        # Mock behavior: Simple mathematical smoothing
        # Simulating a denoiser passing over the float data
        denoised = frame * 0.95 + 0.05
        ctx['frame'] = np.clip(denoised, 0.0, 1.0)

    ctx['metadata']['denoise_model'] = "stub_optix_v2"
    return ctx

def write_provenance_stage(ctx: Dict[str, Any]) -> Dict[str, Any]:
    """Compiles a cryptographically sound ledger of how this frame was made."""
    ctx['provenance'] = {
        "shot": ctx['shot_name'],
        "timestamp": datetime.datetime.utcnow().isoformat() + "Z",
        "pipeline_version": "1.0.4",
        "input_hash_sha256": ctx.get('input_hash', 'N/A'),
        "models_used": {
            "denoise": ctx['metadata'].get('denoise_model'),
            "roto": ctx['metadata'].get('roto_model')
        },
        "manual_review_required": ctx.get('requires_human_review', False)
    }
    return ctx

def save_exr_stage(ctx: Dict[str, Any]) -> Dict[str, Any]:
    """Writes the float32 EXR and the Provenance sidecar."""
    out_dir = Path(ctx['output_dir'])
    out_dir.mkdir(parents=True, exist_ok=True)

    shot_name = ctx['shot_name']

    # 1. Save Main Frame
    out_frame_path = out_dir / f"{shot_name}_comp.exr"
    if HAS_IMAGEIO:
        # Saving as float32 TIFF to guarantee execution without C-bindings for OpenEXR
        # In production, use extension .exr and format='EXR'
        imageio.imwrite(out_dir / f"{shot_name}_comp.tif", (ctx['frame'] * 255).astype(np.uint8))
    ctx['output_frame_path'] = str(out_frame_path)

    # 2. Save AOVs (Mattes)
    if 'roto_matte' in ctx['aovs']:
        aov_path = out_dir / f"{shot_name}_matte.tif"
        if HAS_IMAGEIO:
            imageio.imwrite(aov_path, (ctx['aovs']['roto_matte'] * 255).astype(np.uint8))

    # 3. Save Provenance Sidecar JSON
    sidecar_path = out_dir / f"{shot_name}_provenance.json"
    with open(sidecar_path, 'w') as f:
        json.dump(ctx['provenance'], f, indent=4)

    ctx['provenance_path'] = str(sidecar_path)
    return ctx

def qc_check_stage(ctx: Dict[str, Any]) -> Dict[str, Any]:
    """Automated Quality Control: Checks for NaNs, clipping, and metadata integrity."""
    frame = ctx['frame']

    # 1. Math checks
    if np.isnan(frame).any():
        raise ValueError("QC FAILED: NaN values detected in output frame.")

    if np.max(frame) > 1.0 or np.min(frame) < 0.0:
        logging.warning("QC WARNING: Pixel values out of [0, 1] bounds (Clipping).")

    # 2. Metadata checks
    if not os.path.exists(ctx['provenance_path']):
        raise ValueError("QC FAILED: Provenance sidecar missing.")

    logging.info("  [+] QC PASSED: Image math and provenance verified.")
    return ctx


# ==========================================
# 3. OTIO DISPATCHER & WORKER QUEUE
# ==========================================

class FarmDispatcher:
    def __init__(self, pipeline: Pipeline):
        self.pipeline = pipeline
        self.job_queue = []

    def parse_timeline(self, timeline_path: str, output_dir: str):
        """Reads an editorial timeline and creates VFX jobs for every shot."""
        logging.info(f"Parsing Timeline: {timeline_path}")

        if HAS_OTIO and timeline_path.endswith('.otio'):
            timeline = otio.adapters.read_from_file(timeline_path)
            for track in timeline.tracks:
                if track.kind == otio.schema.TrackKind.Video:
                    for clip in track.each_clip():
                        shot_name = clip.name
                        src_path = clip.media_reference.target_url
                        self._add_job(shot_name, src_path, output_dir)
        else:
            # Fallback: Parse a mock JSON timeline
            with open(timeline_path, 'r') as f:
                data = json.load(f)
                for clip in data.get('clips', []):
                    self._add_job(clip['name'], clip['source'], output_dir)

    def _add_job(self, shot_name: str, src_path: str, output_dir: str):
        job_ctx = {
            'shot_name': shot_name,
            'input_path': src_path,
            'output_dir': output_dir,
            'metadata': {}
        }
        self.job_queue.append(job_ctx)
        logging.info(f"Queued Shot: {shot_name}")

    def execute_queue(self):
        """Simulates a Render Farm Worker picking up tasks."""
        logging.info(f"Executing Queue: {len(self.job_queue)} jobs.")
        results = []
        for job in self.job_queue:
            result = self.pipeline.run(job)
            results.append(result)

        successes = len([r for r in results if r.get('status') == 'SUCCESS'])
        logging.info(f"Queue Complete. {successes}/{len(self.job_queue)} successful.")
        return results

# ==========================================
# 4. EXECUTION & DUMMY DATA GENERATION
# ==========================================

def setup_dummy_production_environment():
    """Generates fake camera originals and a timeline so the script can run anywhere."""
    os.makedirs('ingest', exist_ok=True)
    os.makedirs('renders', exist_ok=True)

    # Generate mock camera raw files (TIFF representing EXR)
    if HAS_IMAGEIO:
        noise1 = (np.random.rand(1080, 1920, 3) * 255).astype(np.uint8)
        imageio.imwrite('ingest/sc01_sh010_raw.tif', noise1)
        noise2 = (np.random.rand(1080, 1920, 3) * 255).astype(np.uint8)
        imageio.imwrite('ingest/sc01_sh020_raw.tif', noise2)
    else:
        Path('ingest/sc01_sh010_raw.tif').touch()
        Path('ingest/sc01_sh020_raw.tif').touch()

    # Generate mock timeline
    timeline_data = {
        "clips": [
            {"name": "sc01_sh010", "source": "ingest/sc01_sh010_raw.tif"},
            {"name": "sc01_sh020", "source": "ingest/sc01_sh020_raw.tif"}
        ]
    }
    with open('ingest/sequence_v01.json', 'w') as f:
        json.dump(timeline_data, f)

    logging.info("Test Production Environment Created.")

if __name__ == "__main__":
    # 1. Setup Data
    setup_dummy_production_environment()

    # 2. Assemble the Pipeline Nodes
    vfx_pipeline = Pipeline(
        name="Cinematic_Main_v1",
        stages=[
            ingest_exr_stage,
            ml_denoise_stage,
            ml_roto_stage,
            write_provenance_stage,
            save_exr_stage,
            qc_check_stage
        ]
    )

    # 3. Initialize Dispatcher and Parse Timeline
    farm = FarmDispatcher(pipeline=vfx_pipeline)
    farm.parse_timeline('ingest/sequence_v01.json', output_dir='renders/final_comps')

    # 4. Execute (In production, this is done by Dask, Celery, or Deadline)
    results = farm.execute_queue()

    # 5. Output Verification
    print("\n--- Pipeline Audit Trail ---")
    for res in results:
        if res.get('status') == 'SUCCESS':
            prov_file = res.get('provenance_path')
            if prov_file and os.path.exists(prov_file):
                with open(prov_file, 'r') as f:
                    prov_data = json.load(f)
                    print(f"Shot: {prov_data['shot']}")
                    print(f"  Input Hash: {prov_data['input_hash_sha256']}")
                    print(f"  Requires Manual Roto Check: {prov_data['manual_review_required']}")
                    print(f"  Time: {prov_data['timestamp']}\n")